# Challenge de la Sesión 06 — Del modelo al producto: umbral, app y asistente
*Introduction to Basic Application Development · Introducción al desarrollo de aplicaciones básicas*

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/06-basic-ai-applications/challenge.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/06-basic-ai-applications/challenge.ipynb)

> **Modalidad:** individual · **Tiempo:** 40 minutos en clase + entrega hasta las **23:59 de hoy** (hora de Colombia).
>
> **IA permitida con registro:** puedes usar asistentes de IA, pero debes declararlos en el registro del final; sin
> registro, el componente de reflexión y registro de IA vale 0.0. Eres responsable de todo lo que entregas.
>
> **Micro-sustentaciones:** durante el bloque se eligen estudiantes al azar para explicar su notebook en 2–3 minutos.
> Si no puedes explicar tu entrega, la nota de este challenge queda limitada a 3.0.
>
> **Rúbrica:** ejecución técnica 40% · análisis e interpretación 40% · reflexión y registro de IA 20%
> ([evaluación del curso](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/evaluacion.md)).
>
> **Entrega:** ejecuta todo en orden → `File → Download notebook` → súbelo a e-Aulas como **`S06_<codigo>.ipynb`**.

**Tu caso.** Eres responsable de producto de IA en una empresa ficticia de servicios financieros. Te asignaron un
proceso de servicio al cliente con 10 intenciones de Banking77 y unos costos unitarios. Vas a:

| Tarea | Qué haces | Componente |
|---|---|---|
| 1 | Entrenar tu clasificador y elegir el umbral de **mínimo costo esperado** | Ejecución + análisis |
| 2 | Construir tu app en Gradio con ese umbral y usarla | Ejecución |
| 3 | **Explica y decide:** asistente en Startti (o alternativa en Python) y decisión código vs. no-code | Ejecución + análisis |
| 4 | **Crítica a la IA:** un flujo de UX propuesto por una IA con 3 anti-patrones | Análisis |

**Kaggle:** *Settings → Internet* **On**; la CPU es suficiente. **Startti es opcional:** la Tarea 3 acepta la
alternativa en Python y ambas se califican con la misma rúbrica. Si usas Startti, agrega tu clave en
*Add-ons → Secrets* como `STARTTI_API_KEY` ([guía](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/configuracion-startti.md)).

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S06"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # los modelos y datos se descargan de Hugging Face
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "Sin conexión a internet: este notebook descarga modelos y datos de Hugging Face.\n"
            "Kaggle: en el panel derecho abre Settings (Session options) y activa Internet. "
            "La opción solo aparece si tu cuenta tiene el teléfono verificado (kaggle.com/settings). "
            "Luego vuelve a ejecutar esta celda. En Colab el internet viene activado. "
            "En tu propio computador: revisa tu conexión, o define HF_HUB_OFFLINE=1 si ya tienes los modelos descargados."
        ) from None

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---
ensure("datasets")
ensure("scikit-learn", "sklearn")
ensure("gradio")

In [ ]:
# Seis contextos de servicio al cliente (empresas ficticias), cada uno con 10 intenciones de Banking77
TEMAS = {  # descripción en español de cada intención
    "activate_my_card": "activar la tarjeta", "card_arrival": "llegada de la tarjeta nueva",
    "card_not_working": "tarjeta que no funciona", "lost_or_stolen_card": "tarjeta perdida o robada",
    "compromised_card": "tarjeta comprometida (posible fraude)", "change_pin": "cambiar el PIN",
    "pin_blocked": "PIN bloqueado", "card_swallowed": "cajero que retuvo la tarjeta",
    "declined_card_payment": "pago con tarjeta rechazado", "contactless_not_working": "pago sin contacto que no funciona",
    "cancel_transfer": "cancelar una transferencia", "declined_transfer": "transferencia rechazada",
    "failed_transfer": "transferencia fallida", "pending_transfer": "transferencia pendiente",
    "transfer_not_received_by_recipient": "transferencia que no le llegó al destinatario",
    "transfer_timing": "tiempo que tarda una transferencia", "transfer_fee_charged": "comisión cobrada por una transferencia",
    "beneficiary_not_allowed": "beneficiario no permitido", "transfer_into_account": "transferir dinero a la cuenta",
    "receiving_money": "recibir dinero",
    "automatic_top_up": "recarga automática", "pending_top_up": "recarga pendiente",
    "top_up_by_bank_transfer_charge": "comisión por recargar con transferencia bancaria",
    "top_up_by_card_charge": "comisión por recargar con tarjeta", "top_up_by_cash_or_cheque": "recargar con efectivo o cheque",
    "top_up_failed": "recarga fallida", "top_up_limits": "límites de recarga", "top_up_reverted": "recarga revertida",
    "topping_up_by_card": "recargar con tarjeta", "verify_top_up": "verificación de una recarga",
    "verify_my_identity": "verificar mi identidad", "unable_to_verify_identity": "no logro verificar mi identidad",
    "why_verify_identity": "por qué debo verificar mi identidad", "verify_source_of_funds": "verificar el origen de los fondos",
    "edit_personal_details": "cambiar mis datos personales", "passcode_forgotten": "olvidé mi clave de acceso",
    "terminate_account": "cerrar la cuenta", "age_limit": "edad mínima para tener una cuenta",
    "lost_or_stolen_phone": "celular perdido o robado", "country_support": "países donde está disponible el servicio",
    "exchange_rate": "tasa de cambio", "exchange_charge": "comisión por cambio de divisas",
    "exchange_via_app": "cambiar divisas en la app",
    "card_payment_wrong_exchange_rate": "tasa de cambio incorrecta en un pago con tarjeta",
    "wrong_exchange_rate_for_cash_withdrawal": "tasa de cambio incorrecta en un retiro",
    "fiat_currency_support": "monedas disponibles", "atm_support": "cajeros donde se puede usar la tarjeta",
    "cash_withdrawal_charge": "comisión por retiro en cajero", "declined_cash_withdrawal": "retiro en cajero rechazado",
    "wrong_amount_of_cash_received": "el cajero entregó un monto incorrecto",
    "request_refund": "solicitar un reembolso", "Refund_not_showing_up": "reembolso que no aparece",
    "transaction_charged_twice": "cobro duplicado", "card_payment_not_recognised": "pago con tarjeta no reconocido",
    "direct_debit_payment_not_recognised": "débito automático no reconocido",
    "extra_charge_on_statement": "cargo extra en el extracto", "card_payment_fee_charged": "comisión en un pago con tarjeta",
    "pending_card_payment": "pago con tarjeta pendiente", "reverted_card_payment?": "pago con tarjeta revertido",
    "cash_withdrawal_not_recognised": "retiro en cajero no reconocido",
}
CASOS = {
    "andes_tarjetas": {
        "empresa": "Andes Bank", "tipo": "banco digital", "proceso": "soporte de tarjetas",
        "intents": ["activate_my_card", "card_arrival", "card_not_working", "lost_or_stolen_card", "compromised_card",
                    "change_pin", "pin_blocked", "card_swallowed", "declined_card_payment", "contactless_not_working"],
        "sensibles": ["lost_or_stolen_card", "compromised_card"],
        "mensaje_es": "Mi tarjeta nueva no ha llegado y ya pasaron dos semanas"},
    "andino_transferencias": {
        "empresa": "Banco Andino", "tipo": "banco", "proceso": "transferencias",
        "intents": ["cancel_transfer", "declined_transfer", "failed_transfer", "pending_transfer",
                    "transfer_not_received_by_recipient", "transfer_timing", "transfer_fee_charged",
                    "beneficiary_not_allowed", "transfer_into_account", "receiving_money"],
        "sensibles": ["cancel_transfer", "transfer_not_received_by_recipient"],
        "mensaje_es": "Hice una transferencia ayer y la otra persona todavía no la recibe"},
    "condor_recargas": {
        "empresa": "Billetera Cóndor", "tipo": "billetera digital", "proceso": "recargas de saldo",
        "intents": ["automatic_top_up", "pending_top_up", "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
                    "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits", "top_up_reverted",
                    "topping_up_by_card", "verify_top_up"],
        "sensibles": ["top_up_reverted", "verify_top_up"],
        "mensaje_es": "Recargué con mi tarjeta y el saldo no aparece"},
    "guayacan_identidad": {
        "empresa": "Cooperativa Guayacán", "tipo": "cooperativa financiera digital", "proceso": "identidad y cuenta",
        "intents": ["verify_my_identity", "unable_to_verify_identity", "why_verify_identity", "verify_source_of_funds",
                    "edit_personal_details", "passcode_forgotten", "terminate_account", "age_limit",
                    "lost_or_stolen_phone", "country_support"],
        "sensibles": ["lost_or_stolen_phone", "verify_source_of_funds"],
        "mensaje_es": "No puedo verificar mi identidad en la app, me rechaza la foto"},
    "colibri_divisas": {
        "empresa": "Colibrí Viajes", "tipo": "tarjeta prepagada para viajeros", "proceso": "divisas y cajeros",
        "intents": ["exchange_rate", "exchange_charge", "exchange_via_app", "card_payment_wrong_exchange_rate",
                    "wrong_exchange_rate_for_cash_withdrawal", "fiat_currency_support", "atm_support",
                    "cash_withdrawal_charge", "declined_cash_withdrawal", "wrong_amount_of_cash_received"],
        "sensibles": ["wrong_amount_of_cash_received", "card_payment_wrong_exchange_rate"],
        "mensaje_es": "El cajero me entregó menos dinero del que pedí"},
    "novatel_cobros": {
        "empresa": "NovaTel Pagos", "tipo": "billetera del operador NovaTel", "proceso": "cobros y reembolsos",
        "intents": ["request_refund", "Refund_not_showing_up", "transaction_charged_twice", "card_payment_not_recognised",
                    "direct_debit_payment_not_recognised", "extra_charge_on_statement", "card_payment_fee_charged",
                    "pending_card_payment", "reverted_card_payment?", "cash_withdrawal_not_recognised"],
        "sensibles": ["card_payment_not_recognised", "direct_debit_payment_not_recognised",
                      "cash_withdrawal_not_recognised"],
        "mensaje_es": "Me cobraron dos veces la misma compra"},
}
COSTO_AUTO = 0.05  # USD por respuesta automática (cómputo y plataforma) · ilustrativo, igual para todos

CONFIG = {
    "caso": pick(sorted(CASOS)),
    "costo_humano": pick([1.0, 2.0, 3.0]),         # USD por mensaje atendido por una persona · ilustrativo
    "costo_error": pick([10.0, 20.0, 40.0]),       # USD por respuesta automática equivocada · ilustrativo
    "tickets_mes": pick([20_000, 50_000, 120_000]),  # mensajes que recibe el proceso cada mes
}
CASO = CASOS[CONFIG["caso"]]

print("🎯 Tu configuración personal")
print(f"   Empresa (ficticia): {CASO['empresa']} ({CASO['tipo']}) · proceso: {CASO['proceso']}")
print(f"   Intenciones: {', '.join(CASO['intents'])}")
print(f"   Costos unitarios (USD, ilustrativos): automático {COSTO_AUTO} · humano {CONFIG['costo_humano']} · "
      f"error automático {CONFIG['costo_error']}")
print(f"   Volumen: {CONFIG['tickets_mes']:,} mensajes al mes")
print(json.dumps(CONFIG, ensure_ascii=False))

## Tarea 1 — Umbral de mínimo costo esperado (ejecución técnica + análisis e interpretación)

Como en el lab: si la probabilidad más alta del clasificador es **≥ umbral**, el mensaje se responde
automáticamente; si no, pasa a una persona. Con cobertura $c$ (share automático) y exactitud de la parte
automática $a$:

$$\text{Costo por 1.000 mensajes} = 1000\,\big[\,c\,C_{auto} + c\,(1-a)\,C_{error} + (1-c)\,C_{humano}\,\big]$$

1. Ejecuta las celdas: cargan tus 10 intenciones, separan **entrenamiento / validación / prueba** y entrenan
   TF-IDF + regresión logística.
2. Calcula cobertura, exactitud y costo para los umbrales 0.30–0.90 **en validación** y elige el de menor costo.
3. Compara con el umbral teórico $t^* = 1 - (C_{humano} - C_{auto})/C_{error}$, que solo es óptimo si las
   probabilidades están **calibradas**. Revisa la tabla de calibración.
4. Reporta el resultado en **prueba** y compáralo con "todo a una persona" y "todo automático".

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

banking = load_dataset("legacy-datasets/banking77")
NOMBRES = banking["train"].features["label"].names


def filtrar(split, max_por_intencion=None):
    """Mensajes de tus 10 intenciones (opcionalmente, un máximo por intención)."""
    df = banking[split].to_pandas()
    df["intent"] = [NOMBRES[l] for l in df["label"]]
    df = df[df["intent"].isin(CASO["intents"])]
    if max_por_intencion:
        df = df.groupby("intent", group_keys=False).head(max_por_intencion)
    return df.reset_index(drop=True)


MAX_TRAIN = 40 if FAST_DEV_RUN else None   # prueba automática: menos datos, mismo código
MAX_TEST = 15 if FAST_DEV_RUN else None
train_df, test_df = filtrar("train", MAX_TRAIN), filtrar("test", MAX_TEST)
X_tr, X_val, y_tr, y_val = train_test_split(train_df["text"].tolist(), train_df["intent"].to_numpy(),
                                            test_size=0.2, stratify=train_df["intent"], random_state=42)
X_test, y_test = test_df["text"].tolist(), test_df["intent"].to_numpy()
print(f"Entrenamiento {len(X_tr)} · validación {len(X_val)} · prueba {len(X_test)}")

C_REG = 10.0  # ✏️ opcional: regularización (C más alto = probabilidades más extremas)
clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                    LogisticRegression(C=C_REG, max_iter=1000))
clf.fit(X_tr, y_tr)


def confianza_y_acierto(X, y):
    """Probabilidad más alta y si la intención predicha es la correcta."""
    P = clf.predict_proba(X)
    return P.max(axis=1), clf.classes_[P.argmax(axis=1)] == y


conf_val, ok_val = confianza_y_acierto(X_val, y_val)
conf_test, ok_test = confianza_y_acierto(X_test, y_test)
print(f"Exactitud (sin umbral) · validación {ok_val.mean():.3f} · prueba {ok_test.mean():.3f}")

In [ ]:
def tabla_enrutamiento(conf, ok, umbrales, costo_auto, costo_humano, costo_error):
    """Cobertura, exactitud de la parte automática y costo esperado por 1.000 mensajes para cada umbral."""
    filas = []
    for u in umbrales:
        auto = conf >= u
        cobertura = auto.mean()
        exactitud_auto = ok[auto].mean() if auto.any() else float("nan")
        tasa_error = (auto & ~ok).mean()  # respuestas automáticas equivocadas, sobre TODOS los mensajes
        costo = 1000 * (cobertura * costo_auto + tasa_error * costo_error + (1 - cobertura) * costo_humano)
        filas.append({"umbral": float(u), "cobertura": cobertura, "exactitud_auto": exactitud_auto,
                      "errores_por_1000": 1000 * tasa_error, "costo_por_1000": costo})
    return pd.DataFrame(filas)


COSTOS = (COSTO_AUTO, CONFIG["costo_humano"], CONFIG["costo_error"])
UMBRALES = np.round(np.arange(0.30, 0.91, 0.05), 2)  # ✏️ opcional: amplía la grilla si tu óptimo queda en un extremo

tabla_val = tabla_enrutamiento(conf_val, ok_val, UMBRALES, *COSTOS)
UMBRAL_ELEGIDO = float(tabla_val.loc[tabla_val["costo_por_1000"].idxmin(), "umbral"])  # ✏️ si eliges otro, justifícalo
UMBRAL_TEORICO = 1 - (CONFIG["costo_humano"] - COSTO_AUTO) / CONFIG["costo_error"]
print(tabla_val.round(3).to_string(index=False))
print(f"\nUmbral de mínimo costo en validación: {UMBRAL_ELEGIDO:.2f} · umbral teórico (si calibrado): {UMBRAL_TEORICO:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(tabla_val["umbral"], tabla_val["costo_por_1000"], marker="o", label="costo esperado (validación)")
ax.axhline(1000 * CONFIG["costo_humano"], ls="--", c="gray", label="todo a una persona")
ax.axvline(UMBRAL_ELEGIDO, ls=":", c="C3", label=f"umbral elegido {UMBRAL_ELEGIDO:.2f}")
ax.set(xlabel="Umbral de confianza", ylabel="USD por 1.000 mensajes", title=f"{CASO['empresa']} · {CASO['proceso']}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Calibración: ¿la confianza del modelo coincide con su exactitud real?
calib = pd.DataFrame({"confianza": conf_val, "acierto": ok_val})
calib["rango"] = pd.cut(calib["confianza"], bins=[0, 0.3, 0.5, 0.7, 0.9, 1.0], include_lowest=True)
print(calib.groupby("rango", observed=True).agg(
    mensajes=("acierto", "size"), confianza_media=("confianza", "mean"), exactitud=("acierto", "mean")).round(3))

# Resultado en prueba con el umbral elegido en validación, y dos políticas de referencia
fila_test = tabla_enrutamiento(conf_test, ok_test, [UMBRAL_ELEGIDO], *COSTOS).iloc[0]
todo_auto = tabla_enrutamiento(conf_test, ok_test, [0.0], *COSTOS).iloc[0]
COSTO_TODO_HUMANO = 1000 * CONFIG["costo_humano"]
AHORRO_MENSUAL = (COSTO_TODO_HUMANO - fila_test["costo_por_1000"]) * CONFIG["tickets_mes"] / 1000
print(f"\nPrueba · umbral {UMBRAL_ELEGIDO:.2f}: cobertura {fila_test['cobertura']:.1%} · exactitud automática "
      f"{fila_test['exactitud_auto']:.1%} · costo {fila_test['costo_por_1000']:.0f} USD por 1.000")
print(f"Referencias · todo a una persona: {COSTO_TODO_HUMANO:.0f} USD · todo automático: {todo_auto['costo_por_1000']:.0f} USD por 1.000")
print(f"Ahorro mensual estimado frente a 'todo a una persona' ({CONFIG['tickets_mes']:,} mensajes): {AHORRO_MENSUAL:,.0f} USD")

**Tu respuesta:**

1. ¿Qué umbral elegiste y qué cobertura, exactitud automática y costo por 1.000 mensajes obtuviste en **prueba**?
   Compáralo con "todo a una persona" y "todo automático", y tradúcelo al ahorro (o pérdida) mensual de tu caso.
2. ¿Por qué tu umbral empírico es distinto del teórico $t^*$? Usa la tabla de calibración para explicarlo.
3. ¿Recomendarías lanzar la automatización con estos números? Menciona un riesgo o un supuesto de tus costos.

*(Escribe aquí, citando tus números.)*

## Tarea 2 — Tu app en Gradio (ejecución técnica)

1. **Personaliza los tres textos** de la siguiente celda aplicando las pautas de Amershi et al. (2019):
   G1 (qué puede hacer el sistema), G2 (qué tan bien lo hace) y G10 (acotar el servicio en caso de duda).
2. Ejecuta la celda de la app. Fuera del modo de prueba se abre con `share=True`, que imprime un **enlace
   público temporal**. Usa la app con **al menos 3 mensajes propios en inglés** (incluye uno ambiguo) y mueve el
   control del umbral.
3. La celda siguiente imprime el registro de interacciones: es tu evidencia.

In [ ]:
# ✏️ Personaliza estos textos (G1, G2 y G10). Puedes usar {intent}, {tema}, {conf} y {umbral}.
TEXTO_BIENVENIDA = (f"**{CASO['empresa']} · asistente de {CASO['proceso']} (prototipo con IA).** "
                    f"Leo tu mensaje en inglés y lo clasifico en uno de {len(CASO['intents'])} temas. "
                    "Si no estoy suficientemente seguro, lo envío a una persona.")
TEXTO_AUTO = "✅ **Respuesta automática** · tema: {tema} (`{intent}`) · confianza {conf:.0%}"
TEXTO_HUMANO = ("🙋 **Lo revisará una persona** · mi mejor hipótesis es {tema} ({conf:.0%}), "
                "por debajo del umbral de {umbral:.0%}.")

REGISTRO_APP = []  # cada mensaje enrutado: evidencia de uso de tu app


def enrutar(mensaje, umbral=UMBRAL_ELEGIDO, origen="notebook"):
    """Clasifica un mensaje y decide: respuesta automática o persona."""
    mensaje = (mensaje or "").strip()
    if not mensaje:
        return {}, "Escribe un mensaje."
    t0 = time.perf_counter()
    P = clf.predict_proba([mensaje])[0]
    latencia_ms = 1000 * (time.perf_counter() - t0)
    orden = np.argsort(P)[::-1][:3]
    top3 = {clf.classes_[i]: float(P[i]) for i in orden}
    intent, conf = clf.classes_[orden[0]], float(P[orden[0]])
    decision = "automática" if conf >= umbral else "persona"
    plantilla = TEXTO_AUTO if decision == "automática" else TEXTO_HUMANO
    texto = plantilla.format(intent=intent, tema=TEMAS[intent], conf=conf, umbral=umbral)
    REGISTRO_APP.append({"hora": time.strftime("%H:%M:%S"), "origen": origen, "mensaje": mensaje, "intent": intent,
                         "confianza": round(conf, 4), "umbral": float(umbral), "decision": decision,
                         "latencia_ms": round(latencia_ms, 1)})
    return top3, texto


# Mensajes de prueba: 4 de tu conjunto de prueba (según tu semilla), uno fuera de alcance y uno en español
_sel = np.random.default_rng(PERSONAL_SEED).choice(len(test_df), size=4, replace=False)
MENSAJES_PRUEBA = [test_df.loc[int(i), "text"] for i in _sel] + [
    "Can you recommend a good restaurant near the office?", CASO["mensaje_es"]]
for m in MENSAJES_PRUEBA:
    print(f"👤 {m}\n   {enrutar(m)[1]}")

In [ ]:
import gradio as gr

with gr.Blocks(title=f"S06 · {CASO['empresa']}") as app:
    gr.Markdown(f"# {CASO['empresa']} · {CASO['proceso']}\n{TEXTO_BIENVENIDA}")
    with gr.Row():
        with gr.Column():
            entrada = gr.Textbox(label="Mensaje del cliente (en inglés)", lines=3)
            control_umbral = gr.Slider(0.0, 1.0, value=UMBRAL_ELEGIDO, step=0.05, label="Umbral de confianza")
            boton = gr.Button("Enrutar", variant="primary")
        with gr.Column():
            salida_top3 = gr.Label(num_top_classes=3, label="Top-3 intenciones")
            salida_decision = gr.Markdown()
    # ✏️ opcional: agrega aquí botones de retroalimentación (👍/👎) como en el lab
    boton.click(lambda m, u: enrutar(m, u, origen="app"), inputs=[entrada, control_umbral],
                outputs=[salida_top3, salida_decision])

if FAST_DEV_RUN:
    print("Prueba automática: la app se construye, pero no se lanza.")
else:
    app.launch(share=True)

In [ ]:
registro = pd.DataFrame(REGISTRO_APP)
print(f"Interacciones registradas: {len(registro)} · desde la app: {int((registro['origen'] == 'app').sum())}")
registro.tail(12)

**Tu respuesta:**

- Pega el **enlace público** que imprimió tu app (es temporal; basta con que exista cuando la uses).
- Describe **una interacción** de tu registro en la que enviar el mensaje a una persona fue la decisión correcta,
  y **una** en la que el umbral se equivocó (o explica por qué no encontraste ninguna).

*(Escribe aquí.)*

## Tarea 3 — Explica y decide: ¿código o no-code para tu caso? (ejecución técnica + análisis e interpretación)

Construye el **asistente conversacional** de tu caso por uno de estos dos caminos (ambos valen lo mismo):

- **Camino A · Startti (no-code).** En [app.startti.ai](https://app.startti.ai) crea un agente con el flujo mínimo
  (entrada de chat → modelo de lenguaje → salida de chat) y pega las instrucciones que imprime la celda de abajo.
  Pruébalo en la vista previa, **publícalo**, guarda tu API key en *Kaggle Secrets* como `STARTTI_API_KEY`,
  vuelve a ejecutar la celda del cliente y pega el ID del agente en `AGENT_ID`.
  Guía: [configuracion-startti.md](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/configuracion-startti.md).
- **Camino B · Python (código).** Completa el diccionario `RESPUESTAS` con **al menos 4 respuestas reales**
  (reemplaza los borradores). El asistente usa tu clasificador, tu umbral y escala a una persona los temas
  sensibles, los de baja confianza y los que aún no tienen respuesta aprobada.

Luego compara ambos enfoques **para tu caso** y decide. Si hiciste el camino B, llena la columna de Startti con
lo que viste en el lab y en la clase.

In [ ]:
# --- Startti ADP client ---
import os, requests
STARTTI_BASE_URL = "https://api.startti.ai"

def get_secret(name):
    """Read a secret from Kaggle Secrets, Colab userdata or an environment variable."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

STARTTI_API_KEY = get_secret("STARTTI_API_KEY")
_STARTTI_HINTS = {401: "invalid or revoked API key", 402: "plan limit reached",
                  403: "the key cannot access this agent", 404: "agent not found — check AGENT_ID",
                  409: "the agent is not published — publish it in ADP first"}

def startti_run(agent_id, prompt, session_key=None, timeout_s=120):
    """Send one message to a published Startti agent and return its text answer."""
    if not STARTTI_API_KEY:
        raise RuntimeError("STARTTI_API_KEY not found. Kaggle: Add-ons → Secrets. Colab: Secrets (key icon). "
                           "On your own computer: set the STARTTI_API_KEY environment variable before starting Jupyter.")
    body = {"agentId": agent_id, "prompt": prompt}
    if session_key:
        body["sessionKey"] = session_key
    r = requests.post(f"{STARTTI_BASE_URL}/v1/run", json=body, timeout=timeout_s,
                      headers={"Authorization": f"Bearer {STARTTI_API_KEY}"})
    if r.status_code >= 400:
        try:
            err = r.json()
        except ValueError:
            err = {"message": r.text[:300]}
        hint = _STARTTI_HINTS.get(r.status_code, "")
        raise RuntimeError(f"Startti API {r.status_code} ({hint}): {err.get('code', '')} {err.get('message', err)}")
    out = r.json().get("output", {})
    if out.get("errorText"):
        raise RuntimeError(f"Agent error: {out['errorText']}")
    if out.get("suspended"):
        questions = "; ".join(q.get("text", "") for q in out.get("suspendQuestions", []))
        return f"[AGENT ASKED] {questions}"
    return out.get("text", "")

print("Startti key found ✅" if STARTTI_API_KEY else "No STARTTI_API_KEY — Startti cells will be skipped.")
# --- end of Startti client ---

In [ ]:
# Instrucciones del sistema para tu agente de Startti (camino A): cópialas en el nodo del modelo de lenguaje
temas_caso = "; ".join(TEMAS[i] for i in CASO["intents"])
temas_sensibles = "; ".join(TEMAS[i] for i in CASO["sensibles"])
INSTRUCCIONES_AGENTE = f"""Eres el asistente virtual de {CASO['empresa']}, una empresa ficticia ({CASO['tipo']}).
Objetivo: ayudar a los clientes con {CASO['proceso']}. Temas que cubres: {temas_caso}.
Tono: cordial, claro y breve (máximo 4 frases). Responde en el idioma del cliente.
Reglas:
- Preséntate como asistente virtual (IA); nunca digas que eres una persona.
- Si no estás seguro o el tema no está en la lista, dilo y ofrece hablar con un asesor humano.
- Temas sensibles ({temas_sensibles}): no los resuelvas; pasa de inmediato a un asesor humano.
- Nunca pidas contraseñas, números completos de tarjeta ni documentos de identidad.
- No inventes plazos, montos, comisiones ni políticas: si no los conoces, ofrece un asesor."""
print(INSTRUCCIONES_AGENTE)

In [ ]:
AGENT_ID = ""  # ✏️ camino A: pega aquí el ID de tu agente publicado

# ✏️ Camino B: reemplaza al menos 4 borradores por respuestas reales (claras, sin prometer lo que no controlas)
RESPUESTAS = {intent: f"(Borrador) Respuesta sobre {TEMAS[intent]}." for intent in CASO["intents"]}
PROB_MIN_SENSIBLE = 0.20  # si un tema sensible tiene al menos esta probabilidad, pasa directo a una persona
TEXTO_ESCALAMIENTO = ("No estoy seguro de poder responderte bien, así que le paso tu mensaje a un asesor humano. "
                      "También puedes reformular tu pregunta.")


def asistente_python(mensaje, umbral=UMBRAL_ELEGIDO):
    """Camino B: clasificador + umbral + respuestas aprobadas + escalamiento. Devuelve (texto, escaló)."""
    P = clf.predict_proba([mensaje])[0]
    intent, conf = clf.classes_[P.argmax()], float(P.max())
    prob = dict(zip(clf.classes_, P))
    sensibles = [s for s in CASO["sensibles"] if prob[s] >= PROB_MIN_SENSIBLE]
    if sensibles:  # primero la seguridad, sin importar el umbral
        return f"🔒 Tu caso ({TEMAS[sensibles[0]]}) necesita a un asesor humano. Te estoy comunicando.", True
    respuesta = RESPUESTAS.get(intent, "")
    if conf < umbral or not respuesta or respuesta.startswith("(Borrador)"):
        return TEXTO_ESCALAMIENTO, True
    return f"{respuesta}\n\n_(Asistente virtual · tema: {TEMAS[intent]} · confianza {conf:.0%})_", False


if STARTTI_API_KEY and AGENT_ID:
    CAMINO_T3 = "startti"
    print(f"✅ Camino A: se usará tu agente de Startti ({AGENT_ID}) y también el asistente en Python para comparar.")
else:
    CAMINO_T3 = "python"
    print("ℹ️ Startti no está configurado (falta STARTTI_API_KEY o AGENT_ID): se usa el camino B (Python).\n"
          "   Para el camino A: publica tu agente, guarda la clave en Kaggle Secrets como STARTTI_API_KEY,\n"
          "   vuelve a ejecutar la celda del cliente y pega el ID en AGENT_ID.")

In [ ]:
# Comparación con los mismos 6 mensajes de prueba
filas = []
for i, m in enumerate(MENSAJES_PRUEBA):
    t0 = time.perf_counter()
    texto_py, escalo = asistente_python(m)
    fila = {"mensaje": m, "python": texto_py, "python_escalo": escalo,
            "python_ms": round(1000 * (time.perf_counter() - t0), 1)}
    if CAMINO_T3 == "startti":
        t0 = time.perf_counter()
        try:
            fila["startti"] = startti_run(AGENT_ID, m, session_key=f"s06-{STUDENT_ID.strip()}-{i}")
        except Exception as exc:
            fila["startti"] = f"ERROR: {exc}"
        fila["startti_ms"] = round(1000 * (time.perf_counter() - t0), 1)
    filas.append(fila)
comparacion = pd.DataFrame(filas)
ESCALADOS_PYTHON = float(comparacion["python_escalo"].mean())
LATENCIA_PYTHON_MS = float(comparacion["python_ms"].median())
LATENCIA_STARTTI_MS = float(comparacion["startti_ms"].median()) if "startti_ms" in comparacion else None
print(f"Python · escaló {ESCALADOS_PYTHON:.0%} de los mensajes · latencia mediana {LATENCIA_PYTHON_MS:.1f} ms")
if LATENCIA_STARTTI_MS is not None:
    print(f"Startti · latencia mediana {LATENCIA_STARTTI_MS:.0f} ms")
pd.set_option("display.max_colwidth", 120)
comparacion

In [ ]:
# El asistente como chat (Gradio ChatInterface): Startti si está configurado, si no, Python
_sesion = {"clave": None}


def chat(mensaje, historial):
    if not historial:
        _sesion["clave"] = f"s06-chat-{STUDENT_ID.strip()}-{int(time.time())}"
    if CAMINO_T3 == "startti":
        try:
            return startti_run(AGENT_ID, mensaje, session_key=_sesion["clave"])
        except Exception as exc:
            print("Error de Startti:", exc)
            return "Lo siento, el asistente no está disponible en este momento. Un asesor humano te contactará."
    return asistente_python(mensaje)[0]


chat_app = gr.ChatInterface(chat, title=f"{CASO['empresa']} · asistente virtual (prototipo)",
                            description=TEXTO_BIENVENIDA)
if FAST_DEV_RUN:
    print("Prueba automática: el chat se construye, pero no se lanza.")
else:
    chat_app.launch(share=True)

**Tu respuesta:**

Completa la tabla con **tus** observaciones y números (latencias, % escalado, qué pasó con el mensaje en español
y con el mensaje fuera de alcance).

| Criterio | Código (Python + Gradio) | No-code (Startti) |
|---|---|---|
| Tiempo para tener la primera versión | | |
| Quién puede mantenerlo en tu empresa | | |
| Control del escalamiento a una persona | | |
| Idiomas (mensaje en español) | | |
| Latencia medida | | |
| Costo por mensaje (qué lo determina) | | |
| Registro, evaluación y privacidad de los datos | | |

**Decisión (3–5 líneas):** para *tu* caso, ¿qué lanzarías primero, código o no-code, y por qué? ¿Qué harías con el
otro enfoque? Cita al menos dos de tus números.

*(Escribe aquí.)*

## Tarea 4 — Crítica a la IA (análisis e interpretación)

Le pediste a un asistente de IA un flujo de experiencia de usuario para el chatbot de tu caso. Esta fue su
propuesta. Contiene **exactamente 3 anti-patrones** de UX para IA; los demás puntos son prácticas correctas.

> **Propuesta del asistente de IA**
>
> 1. **Bienvenida:** "¡Hola! Soy Valentina, tu asesora personal. Puedo resolver cualquier solicitud que tengas, al
>    instante."
> 2. **Confirmación:** antes de ejecutar una acción irreversible (por ejemplo, bloquear una tarjeta o cancelar una
>    transferencia), el asistente explica qué va a pasar y pide confirmación.
> 3. **Baja confianza:** si la confianza del clasificador está por debajo del umbral, el asistente responde igual
>    con la intención más probable y no se lo dice al cliente, para que la conversación sea fluida.
> 4. **Retroalimentación:** cada respuesta tiene 👍/👎 y un campo opcional "¿qué estuvo mal?"; el equipo revisa los
>    comentarios cada semana para corregir respuestas y reentrenar el clasificador.
> 5. **Paso a una persona:** para reducir costos, el botón "Hablar con un asesor" solo aparece después de tres
>    respuestas fallidas, y el cliente debe escribir exactamente la palabra AGENTE.

Para cada punto: di si es un anti-patrón, qué pauta viola o cumple y, si es un anti-patrón, reescríbelo.

**Tu respuesta:**

| # | ¿Anti-patrón? (sí/no) | ¿Qué pauta de Amershi et al. (2019) o del People + AI Guidebook viola o cumple? | Corrección (texto o regla nueva) |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

Explica en 2–3 líneas cuál de los anti-patrones tendría **mayor costo** en tu caso y por qué, usando tus costos
unitarios.

In [ ]:
def _r(x):
    return None if x is None else round(float(x), 4)


RESULTS = {
    "caso": CONFIG["caso"],
    "exactitud_val": _r(ok_val.mean()),
    "exactitud_test": _r(ok_test.mean()),
    "umbral_teorico": _r(UMBRAL_TEORICO),
    "umbral_elegido": _r(UMBRAL_ELEGIDO),
    "costo_1000_val": _r(tabla_val["costo_por_1000"].min()),
    "cobertura_test": _r(fila_test["cobertura"]),
    "exactitud_auto_test": _r(fila_test["exactitud_auto"]),
    "costo_1000_test": _r(fila_test["costo_por_1000"]),
    "costo_1000_todo_humano": _r(COSTO_TODO_HUMANO),
    "costo_1000_todo_auto_test": _r(todo_auto["costo_por_1000"]),
    "ahorro_mensual_usd": _r(AHORRO_MENSUAL),
    "interacciones_app": int((pd.DataFrame(REGISTRO_APP)["origen"] == "app").sum()),
    "camino_t3": CAMINO_T3,
    "escalados_python_t3": _r(ESCALADOS_PYTHON),
    "latencia_python_ms": _r(LATENCIA_PYTHON_MS),
    "latencia_startti_ms": _r(LATENCIA_STARTTI_MS),
}
RESULTS

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))